# Imports de librerias


In [0]:

from pyspark.sql.functions import col, to_timestamp, round

# Lectura de la Tabla Bronce olist_order_items

In [0]:
# creo el df con la tabla bronce de olist_order_items
df = spark.table("`catalog_brazilian-e-commerce`.bronze.olist_order_items_dataset")

In [0]:
df.display()

#Transformaciones

In [0]:

df = (
    df
    # Tipos
    .withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date")))
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double"))

    # Limpieza
    .dropDuplicates()
    .dropna(subset=["order_id", "product_id", "seller_id"])

    # Columna derivada

    .withColumn(
        "total_item_value",
        round(col("price") + col("freight_value"), 2)
    )

)


In [0]:
df.display()

# Crear la tabla Silver de olist_order_items


In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("`catalog_brazilian-e-commerce`.silver.olist_order_items")

In [0]:
%sql
select * from `catalog_brazilian-e-commerce`.silver.olist_order_items;